# Regularization in Practice: Closing the Generalization Gap

Companion notebook for the [Regularization and Generalization](https://codemia.ai/courses/deep_learning_foundations/regularization_and_generalization) lesson on Codemia.

**What you will build**: an intentionally-overfitting MLP, then watch dropout and weight decay close the train/val gap one knob at a time.

*Karpathy's makemore Part 3 touches on this, but this notebook goes deeper on the ablations.*

**Time**: ~30 minutes  
**GPU required**: no

## Setup

In [ ]:
%pip install -q torch matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using {device}')

## Concept recap

Overfitting is what happens when a high-capacity model memorizes noise in the training set. You detect it by watching the **generalization gap** — the difference between training loss and validation loss. A healthy model's curves track each other; an overfit model's train loss goes to zero while val loss climbs.

We will force overfitting on purpose (wide model, tiny dataset), then introduce regularization in three steps:
1. Dropout (randomly zero activations during training)
2. Weight decay (L2 penalty on weights)
3. Both together

After each, we re-plot the gap.

## Implementation

### Step 1 — A tiny dataset and a fat model

Only 200 training examples + 1000 validation examples. The MLP is intentionally over-parameterized relative to the dataset.

In [ ]:
def make_blobs(n, d=20, n_classes=4):
    # n points in R^d, drawn from n_classes Gaussian blobs. Linearly separable-ish;
    # a big MLP will happily memorize the training points including noise.
    centers = torch.randn(n_classes, d) * 2.0
    y = torch.randint(0, n_classes, (n,))
    X = centers[y] + torch.randn(n, d) * 0.7
    return X, y

# Deliberately tiny training set — 200 points for a model that can fit far more.
X_train, y_train = make_blobs(200)
X_val, y_val = make_blobs(1000)
X_train, y_train = X_train.to(device), y_train.to(device)
X_val, y_val = X_val.to(device), y_val.to(device)
print('train:', X_train.shape, '  val:', X_val.shape)

In [ ]:
class FatMLP(nn.Module):
    def __init__(self, d_in=20, hidden=512, n_classes=4, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, hidden),
            nn.ReLU(),
            # Dropout after activation. During eval() it becomes a no-op.
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_classes),
        )
    def forward(self, x):
        return self.net(x)

tmp = FatMLP()
print('params:', sum(p.numel() for p in tmp.parameters()), '  training examples:', len(X_train))

### Step 2 — Train loop that tracks both train and val loss

In [ ]:
def train(dropout=0.0, weight_decay=0.0, epochs=300, lr=1e-2):
    # Fresh model each call so we always start from the same init (seeded outside).
    torch.manual_seed(42)
    model = FatMLP(dropout=dropout).to(device)
    # weight_decay on the optimizer = L2 penalty on ALL params.
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    hist = {'train': [], 'val': []}
    for epoch in range(epochs):
        # --- train step ---
        model.train()
        logits = model(X_train)
        loss = F.cross_entropy(logits, y_train)
        opt.zero_grad()
        loss.backward()
        opt.step()

        # --- eval step (dropout off) ---
        model.eval()
        with torch.no_grad():
            # Re-evaluate train loss in eval mode so dropout doesn't distort the comparison.
            train_eval_loss = F.cross_entropy(model(X_train), y_train).item()
            val_loss = F.cross_entropy(model(X_val), y_val).item()
        hist['train'].append(train_eval_loss)
        hist['val'].append(val_loss)

        if epoch % 50 == 0:
            gap = val_loss - train_eval_loss
            print(f'  epoch {epoch:4d}  train={train_eval_loss:.3f}  val={val_loss:.3f}  gap={gap:+.3f}')
    return hist

## Experiment: four configurations

In [ ]:
configs = {
    'baseline (no reg)':    dict(dropout=0.0, weight_decay=0.0),
    'dropout=0.3':          dict(dropout=0.3, weight_decay=0.0),
    'weight_decay=1e-2':    dict(dropout=0.0, weight_decay=1e-2),
    'dropout+weight_decay': dict(dropout=0.3, weight_decay=1e-2),
}

results = {}
for name, kw in configs.items():
    print(f'\n=== {name} ===')
    results[name] = train(**kw)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True, sharey=True)
for ax, (name, hist) in zip(axes.flat, results.items()):
    ax.plot(hist['train'], label='train')
    ax.plot(hist['val'], label='val')
    # Shade the generalization gap so the reader can literally see it shrink.
    ax.fill_between(range(len(hist['train'])), hist['train'], hist['val'],
                    alpha=0.2, color='red')
    ax.set_title(name)
    ax.legend()
    ax.grid(alpha=0.3)
fig.suptitle('Generalization gap (shaded) shrinks as regularization is added')
plt.tight_layout(); plt.show()

In [ ]:
# Numerical summary — final val loss and final gap side by side.
print(f"{'config':25s}  {'final train':>12s}  {'final val':>12s}  {'gap':>8s}")
for name, hist in results.items():
    t = hist['train'][-1]
    v = hist['val'][-1]
    print(f'{name:25s}  {t:12.4f}  {v:12.4f}  {v-t:+8.4f}')

## Variations

### Sweep dropout probability

Regularization has a sweet spot. Too little and you overfit; too much and you underfit. Let's sweep dropout from 0 to 0.7 and plot final val loss.

In [ ]:
dropouts = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
final_val = []
for p in dropouts:
    print(f'dropout={p}')
    h = train(dropout=p, weight_decay=0.0, epochs=200)
    final_val.append(h['val'][-1])

plt.figure(figsize=(6, 3.5))
plt.plot(dropouts, final_val, 'o-')
plt.xlabel('dropout probability'); plt.ylabel('final val loss')
plt.title('Regularization has a sweet spot'); plt.grid(alpha=0.3); plt.show()
best = dropouts[int(torch.tensor(final_val).argmin())]
print(f'best dropout on this dataset: {best}')

## What to try next

- Swap Adam's `weight_decay` for manual L2 (`loss += lambda * sum(p.pow(2).sum() for p in model.parameters())`). Are they equivalent? (Hint: for Adam, no — read the AdamW paper.)
- Add **data augmentation** (e.g. Gaussian noise on inputs) and see how it compares to dropout as regularization.
- Scale the training set from 100 to 10000 points and plot how the optimal dropout drops as data grows.
- Add label smoothing (`F.cross_entropy(..., label_smoothing=0.1)`) as a third regularizer.